In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
df = pd.read_csv('../data/raw/Works Recommended.csv')

In [3]:
df.head()

,Sr. No.,Work category,WORK,State,IDA,Hon'ble Members of Parliament,Constituency,Work description,Recommended date,RECOMMENDED AMOUNT ( ₹ ),Sanction Date
0,1,Normal/Others,WS/\t MP620/2024-2025/133166-Construction of b...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan at Navalgund ...,08-Jul-2024,497185,09-Jul-2024
1,2,Trust and Society,WS/\t MP620/2025-2026/133167-Construction of r...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of College room of CBS Charitabl...,08-Jul-2024,500000,18-Sep-2025
2,3,Trust and Society,WS/\t MP620/2024-2025/133190-Construction of b...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan of Veerashaiv...,08-Jul-2024,450000,23-Sep-2024
3,4,Normal/Others,WS/\t MP620/2025-2026/133191-Construction of b...,Karnataka,HAVERI(DEPUTY COMMISSIONER HAVERI_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Maremma Cultural bhavan at Nee...,08-Jul-2024,1500000,29-May-2025
4,5,Normal/Others,WS/\t MP620/2024-2025/133301-Construction of b...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Cultural Bhavan at Kundagol TQ...,09-Jul-2024,398009,22-Aug-2024


In [10]:
df.tail()

,sr_no,work_category,work,state,ida,hon_ble_members_of_parliament,constituency,work_description,recommended_date,recommended_amount,sanction_date
4996,4997,Normal/Others,WS/MP386/2024-2025/145055-Street lights,Rajasthan,NAGAUR(DISTRICT COLLECTOR NAGAUR_IDA),Shri Hanuman Beniwal,NAGAUR,high mask light in village-khatora near Kesriy...,01-Oct-2024,59780,09-Dec-2024
4997,4998,Normal/Others,WS/MP386/2024-2025/145056-Street lights,Rajasthan,NAGAUR(DISTRICT COLLECTOR NAGAUR_IDA),Shri Hanuman Beniwal,NAGAUR,"High mask light in village khatora,near dhanra...",01-Oct-2024,59780,09-Dec-2024
4998,4999,Normal/Others,WS/MP386/2024-2025/145062-Street lights,Rajasthan,NAGAUR(DISTRICT COLLECTOR NAGAUR_IDA),Shri Hanuman Beniwal,NAGAUR,High mask light in aam -gawad and near residan...,01-Oct-2024,59999,13-Dec-2024
4999,5000,Normal/Others,WS/MP386/2024-2025/145063-Street lights,Rajasthan,NAGAUR(DISTRICT COLLECTOR NAGAUR_IDA),Shri Hanuman Beniwal,NAGAUR,"High mask light in SUNARO KA MOHHALA,Near-mata...",01-Oct-2024,60000,09-Dec-2024
5000,Grand Total,,,,,,,,,,"60,08,51,29,623.08"


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5001 entries, 0 to 5000
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   Sr. No.                        5001 non-null   str  
 1   Work category                  5001 non-null   str  
 2   WORK                           5001 non-null   str  
 3   State                          5001 non-null   str  
 4   IDA                            5001 non-null   str  
 5   Hon'ble Members of Parliament  5001 non-null   str  
 6   Constituency                   5001 non-null   str  
 7   Work description               4987 non-null   str  
 8   Recommended date               5001 non-null   str  
 9   RECOMMENDED AMOUNT   ( ₹ )     5001 non-null   str  
 10  Sanction Date                  4942 non-null   str  
dtypes: str(11)
memory usage: 429.9 KB


In [5]:
df.isnull().sum()

Sr. No.                           0
Work category                     0
WORK                              0
State                             0
IDA                               0
Hon'ble Members of Parliament     0
Constituency                      0
Work description                 14
Recommended date                  0
RECOMMENDED AMOUNT   ( ₹ )        0
Sanction Date                    59
dtype: int64

In [6]:
df.duplicated().sum()

0

In [7]:
df.columns = (
    pd.Index(df.columns)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

In [8]:
df.columns

Index(['sr_no', 'work_category', 'work', 'state', 'ida',
       'hon_ble_members_of_parliament', 'constituency', 'work_description',
       'recommended_date', 'recommended_amount', 'sanction_date'],
      dtype='str')

In [9]:
for col in df.columns:
    if col == "work":
        continue

    df[col] = (
        df[col]
        .astype("string")
        .str.replace("\u00a0", " ", regex=False)
        .str.replace(r"[\t\r\n]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


In [11]:
grand_total_mask = pd.Series(False, index=df.index)

for col in df.columns:
    if col == "work":
        continue

    grand_total_mask |= df[col].str.fullmatch(
        r"grand\s+total",
        case=False,
        na=False
    )

print("Grand Total rows found:", int(grand_total_mask.sum()))

df = df.loc[~grand_total_mask].copy()

Grand Total rows found: 1


In [12]:
df = df.replace(r"^\s*$", pd.NA, regex=True)

print("Rows remaining:", len(df))

Rows remaining: 5000


In [13]:
df["work"] = df["work"].str.replace(
    r"\s*/\s*",
    "/",
    regex=True
)

In [14]:
df["work_id"] = df["work"].str.extract(
    r"(?i)(WS/MP\d+/\d{4}-\d{4}/\d+)",
    expand=False
)
df["work_id"] = df["work_id"].str.upper()

In [15]:
df["work_detail_from_reference"] = df["work"].str.extract(
    r"(?i)^WS/MP\d+/\d{4}-\d{4}/\d+-(.*)$",
    expand=False
).str.strip()

In [16]:
df.head()

,sr_no,work_category,work,state,ida,hon_ble_members_of_parliament,constituency,work_description,recommended_date,recommended_amount,sanction_date,work_id,work_detail_from_reference
0,1,Normal/Others,WS/MP620/2024-2025/133166-Construction of buil...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan at Navalgund ...,08-Jul-2024,497185,09-Jul-2024,WS/MP620/2024-2025/133166,Construction of buildings for community cultur...
1,2,Trust and Society,WS/MP620/2025-2026/133167-Construction of room...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of College room of CBS Charitable...,08-Jul-2024,500000,18-Sep-2025,WS/MP620/2025-2026/133167,Construction of rooms and halls in school and ...
2,3,Trust and Society,WS/MP620/2024-2025/133190-Construction of buil...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan of Veerashaiv...,08-Jul-2024,450000,23-Sep-2024,WS/MP620/2024-2025/133190,Construction of buildings for community cultur...
3,4,Normal/Others,WS/MP620/2025-2026/133191-Construction of buil...,Karnataka,HAVERI(DEPUTY COMMISSIONER HAVERI_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Maremma Cultural bhavan at Nee...,08-Jul-2024,1500000,29-May-2025,WS/MP620/2025-2026/133191,Construction of buildings for community cultur...
4,5,Normal/Others,WS/MP620/2024-2025/133301-Construction of buil...,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Cultural Bhavan at Kundagol TQ...,09-Jul-2024,398009,22-Aug-2024,WS/MP620/2024-2025/133301,Construction of buildings for community cultur...


In [17]:
amount_text = (
    df["recommended_amount"]
    .astype("string")
    .str.replace(",", "", regex=False)
    .str.replace("₹", "", regex=False)
    .str.strip()
)


In [18]:
missing_amount = amount_text.str.upper().isin(
    ["", "NA", "N/A", "NULL", "-"]
)
amount_text = amount_text.mask(missing_amount, pd.NA)

In [19]:
df["recommended_amount"] = pd.to_numeric(
    amount_text,
    errors="coerce"
)

In [20]:
def parse_date_column(series):
    cleaned = series.astype("string").str.strip()

    missing = cleaned.str.upper().isin(
        ["", "NA", "N/A", "NULL", "-"]
    )
    cleaned = cleaned.mask(missing, pd.NA)

    return pd.to_datetime(
        cleaned,
        format="%d-%b-%Y",
        errors="coerce"
    )


In [21]:
for col in ["recommended_date", "sanction_date"]:

    raw_dates = df[col].astype("string").str.strip()
    missing = raw_dates.str.upper().isin(
        ["", "NA", "N/A", "NULL", "-"]
    )

    parsed_dates = parse_date_column(raw_dates)

    df[col] = parsed_dates

In [22]:
df.drop(columns=["work"], inplace=True)

In [23]:
df.head()

,sr_no,work_category,state,ida,hon_ble_members_of_parliament,constituency,work_description,recommended_date,recommended_amount,sanction_date,work_id,work_detail_from_reference
0,1,Normal/Others,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan at Navalgund ...,2024-07-08,497185.0,2024-07-09,WS/MP620/2024-2025/133166,Construction of buildings for community cultur...
1,2,Trust and Society,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of College room of CBS Charitable...,2024-07-08,500000.0,2025-09-18,WS/MP620/2025-2026/133167,Construction of rooms and halls in school and ...
2,3,Trust and Society,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Community Bhavan of Veerashaiv...,2024-07-08,450000.0,2024-09-23,WS/MP620/2024-2025/133190,Construction of buildings for community cultur...
3,4,Normal/Others,Karnataka,HAVERI(DEPUTY COMMISSIONER HAVERI_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Maremma Cultural bhavan at Nee...,2024-07-08,1500000.0,2025-05-29,WS/MP620/2025-2026/133191,Construction of buildings for community cultur...
4,5,Normal/Others,Karnataka,DHARWAD(DEPUTY COMMISSIONER DHARWAR_IDA),Pralhad Venkatesh Joshi,DHARWAD,Construction of Cultural Bhavan at Kundagol TQ...,2024-07-09,398009.0,2024-08-22,WS/MP620/2024-2025/133301,Construction of buildings for community cultur...


In [24]:
df.to_csv('../data/processed/works_recommended_cleaned.csv', index=False)